In [ ]:
from pathlib import Path
import sys

# Make the notebook usable both from repository root and from core/tests.
cwd = Path.cwd().resolve()
candidate_roots = [cwd, *cwd.parents]

for candidate in candidate_roots:
    if (candidate / "core").is_dir():
        workspace_root = candidate
        break
else:
    raise RuntimeError("Could not find repository root containing the 'core' directory.")

if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))

from core.properties import (
    IAPWS97WaterSteamProvider,
    water_steam_props_iapws97,
    )

water = water_steam_props_iapws97(T=293.15, p=101325.0)

assert 995.0 < water.transport.rho < 1000.0, water.transport.rho
assert 0.0009 < water.transport.mu < 0.0011, water.transport.mu
assert 0.58 < water.transport.k < 0.62, water.transport.k
assert 4100.0 < water.transport.cp < 4300.0, water.transport.cp
assert 80000.0 < water.h < 90000.0, water.h

provider = IAPWS97WaterSteamProvider()
props = provider.at(T=293.15, p=101325.0)

assert props.rho == water.transport.rho
assert props.mu == water.transport.mu
assert props.k == water.transport.k
assert props.cp == water.transport.cp

print("IAPWS-IF97 water property smoke test passed.")

In [ ]:
from core.properties import (
    IAPWS97WaterSteamProvider,
    FluidTransportProperties,
    from_internal_fluid_props,
    from_internal_pressure_drop_fluid_props,
    from_outside_fluid_props,
    mean_temperature,
    mean_transport_props,
    to_internal_fluid_props,
    to_internal_pressure_drop_fluid_props,
    to_outside_fluid_props,
)

provider = IAPWS97WaterSteamProvider()

T1 = 293.15
T2 = 333.15
p = 101325.0

T_mean = mean_temperature(T1, T2)
props = mean_transport_props(provider=provider, T1=T1, T2=T2, p=p)

internal_props = to_internal_fluid_props(props)
internal_dp_props = to_internal_pressure_drop_fluid_props(props)
outside_props = to_outside_fluid_props(props)

assert T_mean == 313.15

assert props.rho > 0.0
assert props.mu > 0.0
assert props.k > 0.0
assert props.cp > 0.0

assert internal_props.rho == props.rho
assert internal_props.mu == props.mu
assert internal_props.k == props.k
assert internal_props.cp == props.cp

# InternalPressureDropFluidProps only carries rho and mu (used for pressure drop only)
assert internal_dp_props.rho == props.rho
assert internal_dp_props.mu == props.mu

assert outside_props.rho == props.rho
assert outside_props.mu == props.mu
assert outside_props.k == props.k
assert outside_props.cp == props.cp

props_from_internal = from_internal_fluid_props(internal_props)
props_from_outside = from_outside_fluid_props(outside_props)

assert isinstance(props_from_internal, FluidTransportProperties)
assert isinstance(props_from_outside, FluidTransportProperties)

assert props_from_internal == props
assert props_from_outside == props

print("Property layer integration helper smoke test passed.")


In [ ]:
from core.properties import (
    MoistAirTransportProvider,
    moist_air_transport_props_from_state,
    moist_air_transport_result_from_state,
    moist_air_transport_result_from_t_rh,
    to_outside_fluid_props,
)
from core.psychrometrics import moist_air_state_from_t_rh

air = moist_air_state_from_t_rh(T=293.15, RH=0.50, p=101325.0)

transport_result = moist_air_transport_result_from_state(air)
transport_props = transport_result.props

assert 1.19 < transport_props.rho < 1.21, transport_props.rho
assert 1.75e-5 < transport_props.mu < 1.90e-5, transport_props.mu
assert 0.024 < transport_props.k < 0.027, transport_props.k
assert 1000.0 < transport_props.cp < 1030.0, transport_props.cp

outside_props = to_outside_fluid_props(transport_props)

assert outside_props.rho == transport_props.rho
assert outside_props.mu == transport_props.mu
assert outside_props.k == transport_props.k
assert outside_props.cp == transport_props.cp

transport_result_2 = moist_air_transport_result_from_t_rh(
    T=293.15,
    RH=0.50,
    p=101325.0,
)

assert abs(transport_result_2.props.rho - transport_props.rho) < 1e-12
assert abs(transport_result_2.props.cp - transport_props.cp) < 1e-12

provider = MoistAirTransportProvider.from_t_rh(
    T=293.15,
    RH=0.50,
    p=101325.0,
)

provider_props = provider.at(T=303.15, p=101325.0)

assert provider_props.rho > 0.0
assert provider_props.mu > 0.0
assert provider_props.k > 0.0
assert provider_props.cp > 0.0

print("Moist-air transport property smoke test passed.")